# Computation of the IAA

Axel FELTEN, Mario JAKIMOSKI, Hao MENG, Jinane HARCHAL

# Annotation Task

We perform an annotation in common on the files that were generated by different models for the project.

We chose 8 files for each model, each files has different kind. 

In [501]:
import os
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import json
import numpy as np

# Files Paths

In [502]:
MD = "Mario/deepseek"
MG = "Mario/gemma"
ML = "Mario/llama"
MM = "Mario/mistral"
MN = "Mario/nemotron"
MQ = "Mario/qwen"

chemin_M = [MD, MG, ML, MM, MN, MQ]

AD = "Axel/deepseek"
AG = "Axel/gemma"
AL = "Axel/llama"
AM = "Axel/mistral"
AN = "Axel/nemotron"
AQ = "Axel/qwen"

chemin_A = [AD, AG, AL, AM, AN, AQ]

HD = "Hao/deepseek"
HG = "Hao/gemma"
HL = "Hao/llama"
HM = "Hao/mistral"
HN = "Hao/nemotron"
HQ = "Hao/qwen"

chemin_H = [HD, HG, HL, HM, HN, HQ]

JD = "Jinane/deepseek"
JG = "Jinane/gemma"
JL = "Jinane/llama"
JM = "Jinane/mistral"
JN = "Jinane/nemotron"
JQ = "Jinane/qwen"

chemin_J = [JD, JG, JL, JM, JN, JQ]

file_path_mario = [os.path.join(chemin, element) for chemin in chemin_M for element in os.listdir(chemin)]
file_path_axel = [os.path.join(chemin, element) for chemin in chemin_A for element in os.listdir(chemin)]
file_path_hao = [os.path.join(chemin, element) for chemin in chemin_H for element in os.listdir(chemin)]
file_path_jinane = [os.path.join(chemin, element) for chemin in chemin_J for element in os.listdir(chemin)]
file_path_jinane = [files for files in file_path_jinane if "DS_Store" not in files]

In [503]:
file_path_axel.sort()
file_path_mario.sort()
file_path_hao.sort()
file_path_jinane.sort()

In [504]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

def get_cleaned_data(file_path, json_config):
    # Trouver la clé JSON correspondante (avec gestion de la coquille Qween)
    with open(json_config, 'r') as f:
        config_data = json.load(f)

        path_lower = file_path.replace('\\', '/').lower()
        matched_key = None
    
        for key in config_data.keys():
            comp_key = key.lower().replace('qween', 'qwen')
            parts = comp_key.split('/')
            if parts[0].split('_')[0] in path_lower and parts[-1].replace('.json', '') in path_lower:
                matched_key = key
                break
            
        if not matched_key:
            return None, None

        # Extraction des index depuis le JSON
        raw_values = config_data[matched_key]
        all_indices = []
        for i in range(0, len(raw_values), 2):
            all_indices.extend(range(raw_values[i] - 1, raw_values[i+1]))

        # Chargement et nettoyage
        df = pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig')
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        df['comment'] = df['comment'].astype(str).str.strip()
        df.loc[df['rating'].isna() & (df['comment'] == 'empty'), 'rating'] = 0
    
        # On ne garde que les lignes de l'intervalle
        final_df = df.iloc[all_indices].copy()
    
    return matched_key, final_df

In [505]:
with open('result_kappa_without_subjectivity.csv', 'w') as f:
    f.write("Couple,File_1,File_2,Kappa\n")
    for file_a, file_m, file_h, file_j in zip(file_path_axel, file_path_mario, file_path_hao, file_path_jinane):
        matched_key, cleaned_df_a = get_cleaned_data(file_a, "result_nb_question.json")
        matched_key, cleaned_df_m = get_cleaned_data(file_m, "result_nb_question.json")
        matched_key, cleaned_df_h = get_cleaned_data(file_h, "result_nb_question.json")
        matched_key, cleaned_df_j = get_cleaned_data(file_j, "result_nb_question.json")
        dataframes = [cleaned_df_a, cleaned_df_h, cleaned_df_j, cleaned_df_m]

        for dataframe in dataframes:
            dataframe[dataframe == 1] = 0
            dataframe[dataframe == 6] = 8
            dataframe[dataframe == 7] = 8 
            dataframe.loc[dataframe['rating'].isna(), 'rating'] = 0

        kappa_ma = cohen_kappa_score(cleaned_df_a['rating'], cleaned_df_m['rating'])
        print(f"Kappa: between {file_a} and {file_m}: {kappa_ma:.3f}")
        f.write(f"Mario-Axel,{file_a},{file_m},{kappa_ma:.3f}\n")

        kappa_ha = cohen_kappa_score(cleaned_df_a['rating'], cleaned_df_h['rating'])
        print(f"Kappa: between {file_a} and {file_h}: {kappa_ha:.3f}")
        f.write(f"Hao-Axel,{file_a},{file_h},{kappa_ha:.3f}\n")

        kappa_ja = cohen_kappa_score(cleaned_df_a['rating'], cleaned_df_j['rating'])
        print(f"Kappa: between {file_a} and {file_j}: {kappa_ja:.3f}")
        f.write(f"Jinane-Axel,{file_a},{file_j},{kappa_ja:.3f}\n")
        
        kappa_mj = cohen_kappa_score(cleaned_df_m['rating'], cleaned_df_j['rating'])
        print(f"Kappa: between {file_m} and {file_j}: {kappa_mj:.3f}")
        f.write(f"Mario-Jinane,{file_m},{file_j},{kappa_mj:.3f}\n")

        kappa_hj = cohen_kappa_score(cleaned_df_h['rating'], cleaned_df_j['rating'])
        print(f"Kappa: between {file_h} and {file_j}: {kappa_hj:.3f}")
        f.write(f"Hao-Jinane,{file_h},{file_j},{kappa_hj:.3f}\n")

        kappa_mh = cohen_kappa_score(cleaned_df_m['rating'], cleaned_df_h['rating'])
        print(f"Kappa: between {file_m} and {file_h}: {kappa_mh:.3f}")
        f.write(f"Mario-Hao,{file_m},{file_h},{kappa_mh:.3f}\n")

Kappa: between Axel/deepseek/sheet_10_annotations_FELTEN.csv and Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv: 0.167
Kappa: between Axel/deepseek/sheet_10_annotations_FELTEN.csv and Hao/deepseek/sheet_10_annotations_Hao_Deepseek.csv: 0.219
Kappa: between Axel/deepseek/sheet_10_annotations_FELTEN.csv and Jinane/deepseek/sheet_10_annotations_HARCHAL.csv: 0.326
Kappa: between Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv and Jinane/deepseek/sheet_10_annotations_HARCHAL.csv: 0.215
Kappa: between Hao/deepseek/sheet_10_annotations_Hao_Deepseek.csv and Jinane/deepseek/sheet_10_annotations_HARCHAL.csv: 0.347
Kappa: between Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv and Hao/deepseek/sheet_10_annotations_Hao_Deepseek.csv: 0.405
Kappa: between Axel/deepseek/sheet_13_annotations_FELTEN.csv and Mario/deepseek/sheet_13_annotations_JAKIMOSKI.csv: 0.171
Kappa: between Axel/deepseek/sheet_13_annotations_FELTEN.csv and Hao/deepseek/sheet_13_annotations_Hao_Deepseek.csv: 0.074
Kappa: betw

In [506]:
with open('result_kappa.csv', 'w') as f:
    f.write("Couple,File_1,File_2,Kappa\n")
    for file_a, file_m, file_h, file_j in zip(file_path_axel, file_path_mario, file_path_hao, file_path_jinane):
        matched_key, cleaned_df_a = get_cleaned_data(file_a, "result_nb_question.json")
        matched_key, cleaned_df_m = get_cleaned_data(file_m, "result_nb_question.json")
        matched_key, cleaned_df_h = get_cleaned_data(file_h, "result_nb_question.json")
        matched_key, cleaned_df_j = get_cleaned_data(file_j, "result_nb_question.json")
        dataframes = [cleaned_df_a, cleaned_df_h, cleaned_df_j, cleaned_df_m]

        for dataframe in dataframes:
            dataframe.loc[dataframe['rating'].isna(), 'rating'] = 0

        kappa_ma = cohen_kappa_score(cleaned_df_a['rating'], cleaned_df_m['rating'])
        print(f"Kappa: between {file_a} and {file_m}: {kappa_ma:.3f}")
        f.write(f"Mario-Axel,{file_a},{file_m},{kappa_ma:.3f}\n")

        kappa_ha = cohen_kappa_score(cleaned_df_a['rating'], cleaned_df_h['rating'])
        print(f"Kappa: between {file_a} and {file_h}: {kappa_ha:.3f}")
        f.write(f"Hao-Axel,{file_a},{file_h},{kappa_ha:.3f}\n")

        kappa_ja = cohen_kappa_score(cleaned_df_a['rating'], cleaned_df_j['rating'])
        print(f"Kappa: between {file_a} and {file_j}: {kappa_ja:.3f}")
        f.write(f"Jinane-Axel,{file_a},{file_j},{kappa_ja:.3f}\n")
        
        kappa_mj = cohen_kappa_score(cleaned_df_m['rating'], cleaned_df_j['rating'])
        print(f"Kappa: between {file_m} and {file_j}: {kappa_mj:.3f}")
        f.write(f"Mario-Jinane,{file_m},{file_j},{kappa_mj:.3f}\n")

        kappa_hj = cohen_kappa_score(cleaned_df_h['rating'], cleaned_df_j['rating'])
        print(f"Kappa: between {file_h} and {file_j}: {kappa_hj:.3f}")
        f.write(f"Hao-Jinane,{file_h},{file_j},{kappa_hj:.3f}\n")

        kappa_mh = cohen_kappa_score(cleaned_df_m['rating'], cleaned_df_h['rating'])
        print(f"Kappa: between {file_m} and {file_h}: {kappa_mh:.3f}")
        f.write(f"Mario-Hao,{file_m},{file_h},{kappa_mh:.3f}\n")

Kappa: between Axel/deepseek/sheet_10_annotations_FELTEN.csv and Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv: 0.212
Kappa: between Axel/deepseek/sheet_10_annotations_FELTEN.csv and Hao/deepseek/sheet_10_annotations_Hao_Deepseek.csv: 0.256
Kappa: between Axel/deepseek/sheet_10_annotations_FELTEN.csv and Jinane/deepseek/sheet_10_annotations_HARCHAL.csv: 0.296
Kappa: between Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv and Jinane/deepseek/sheet_10_annotations_HARCHAL.csv: 0.224
Kappa: between Hao/deepseek/sheet_10_annotations_Hao_Deepseek.csv and Jinane/deepseek/sheet_10_annotations_HARCHAL.csv: 0.324
Kappa: between Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv and Hao/deepseek/sheet_10_annotations_Hao_Deepseek.csv: 0.366
Kappa: between Axel/deepseek/sheet_13_annotations_FELTEN.csv and Mario/deepseek/sheet_13_annotations_JAKIMOSKI.csv: 0.170
Kappa: between Axel/deepseek/sheet_13_annotations_FELTEN.csv and Hao/deepseek/sheet_13_annotations_Hao_Deepseek.csv: 0.085
Kappa: betw

# Check the results

In [507]:
df = pd.read_csv("result_kappa.csv")
list_kappa_MA = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Mario-Axel"]
list_kappa_HA = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Hao-Axel"]
list_kappa_JA = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Jinane-Axel"]
list_kappa_MJ = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Mario-Jinane"]
list_kappa_HJ = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Hao-Jinane"]
list_kappa_MH = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Mario-Hao"]

print(f"Means kappa:{"\n"}Mario-Axel:{np.mean(list_kappa_MA):.3f},{"\n"}Hao-Axel:{np.mean(list_kappa_HA):.3f},{"\n"}Jinane-Axel:{np.mean(list_kappa_JA):.3f},{"\n"}Mario-Jinane:{np.mean(list_kappa_MJ):.3f},{"\n"}Hao-Jinane:{np.mean(list_kappa_HJ):.3f},{"\n"}Mario-Hao:{np.mean(list_kappa_MH):.3f}")

Means kappa:
Mario-Axel:0.133,
Hao-Axel:0.126,
Jinane-Axel:0.189,
Mario-Jinane:0.144,
Hao-Jinane:0.129,
Mario-Hao:0.126


In [508]:
df = pd.read_csv("result_kappa_without_subjectivity.csv")
list_kappa_MA_su = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Mario-Axel"]
list_kappa_HA_su = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Hao-Axel"]
list_kappa_JA_su = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Jinane-Axel"]
list_kappa_MJ_su = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Mario-Jinane"]
list_kappa_HJ_su = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Hao-Jinane"]
list_kappa_MH_su = [kappa for kappa, couple in zip(df['Kappa'], df['Couple']) if couple == "Mario-Hao"]

print(f"Means kappa without subjectivity:{"\n"}Mario-Axel:{np.mean(list_kappa_MA_su):.3f},{"\n"}Hao-Axel:{np.mean(list_kappa_HA_su):.3f},{"\n"}Jinane-Axel:{np.mean(list_kappa_JA_su):.3f},{"\n"}Mario-Jinane:{np.mean(list_kappa_MJ_su):.3f},{"\n"}Hao-Jinane:{np.mean(list_kappa_HJ_su):.3f},{"\n"}Mario-Hao:{np.mean(list_kappa_MH_su):.3f}")

Means kappa without subjectivity:
Mario-Axel:0.191,
Hao-Axel:0.192,
Jinane-Axel:0.215,
Mario-Jinane:0.174,
Hao-Jinane:0.183,
Mario-Hao:0.167


In [509]:
Kappa = pd.read_csv("result_kappa.csv")
mean_kappa = np.mean(Kappa['Kappa'])
print(f"Global kappa: {mean_kappa:.3f}")

Global kappa: 0.141


In [510]:
Kappa_Without_Subjectivity = pd.read_csv("result_kappa_without_subjectivity.csv")
mean_kappa_without_subjectivity = np.mean(Kappa_Without_Subjectivity['Kappa'])
print(f"Global kappa without subjectivity: {mean_kappa_without_subjectivity:.3f}")

Global kappa without subjectivity: 0.187


# Check all the questions that wa annotated the same way

The following function able to check where the annotators agreed

In [511]:
def check_same_questions(fileA, fileB):
    result = []
    matched_key, df_a = get_cleaned_data(fileA, "result_nb_question.json")
    matched_key, df_b = get_cleaned_data(fileB, "result_nb_question.json")
    dataframes = [df_a, df_b]
    for dataframe in dataframes:
            dataframe.loc[dataframe['rating'].isna(), 'rating'] = 0
            
    for rateA, rateB, questionA, questionB in zip(df_a['rating'], df_b['rating'], df_a["question_text"], df_b["question_text"]):
        if rateA == rateB:
            result.append([questionA, questionB, rateA, rateB])
    return result

In [512]:
def check_same_questions_without_subjectivity(fileA, fileB):
    result = []
    matched_key, df_a = get_cleaned_data(fileA, "result_nb_question.json")
    matched_key, df_b = get_cleaned_data(fileB, "result_nb_question.json")
    dataframes = [df_a, df_b]
    for dataframe in dataframes:
        dataframe[dataframe == 1] = 0
        dataframe[dataframe == 6] = 8
        dataframe[dataframe == 7] = 8
        dataframe.loc[dataframe['rating'].isna(), 'rating'] = 0
        
    for rateA, rateB, questionA, questionB in zip(df_a['rating'], df_b['rating'], df_a["question_text"], df_b["question_text"]):
        if rateA == rateB:
            result.append([questionA, questionB, rateA, rateB])
    return result

In [513]:
def check_same_questions_without_subjectivity_4(fileA, fileB, fileC, fileD):
    result = []
    matched_key, df_a = get_cleaned_data(fileA, "result_nb_question.json")
    matched_key, df_b = get_cleaned_data(fileB, "result_nb_question.json")
    matched_key, df_c = get_cleaned_data(fileC, "result_nb_question.json")
    matched_key, df_d = get_cleaned_data(fileD, "result_nb_question.json")
    dataframes = [df_a, df_b, df_c, df_d]
    for dataframe in dataframes:
        dataframe[dataframe == 1] = 0
        dataframe[dataframe == 6] = 8
        dataframe[dataframe == 7] = 8
        dataframe.loc[dataframe['rating'].isna(), 'rating'] = 0
        
    for rateA, rateB, rateC, rateD, questionA, questionB, questionC, questionD in zip(df_a['rating'], df_b['rating'], df_c['rating'], df_d['rating'], df_a["question_text"], df_b["question_text"], df_c["question_text"], df_d["question_text"]):
        if rateA == rateB == rateC == rateD:
            result.append([questionA, questionB, questionC, questionD, rateA, rateB, rateC, rateD])
    return result

## Check for all the file when they agreed

In [514]:
for fileA, fileM, fileH, fileJ in zip(file_path_axel, file_path_mario, file_path_hao, file_path_jinane):
    print(f"MA:{fileM} , {len(check_same_questions(fileM, fileA))} , {check_same_questions(fileM, fileA)}")
    print("\n")
    print(f"HA:{fileH} , {len(check_same_questions(fileH, fileA))} , {check_same_questions(fileH, fileA)}")
    print("\n")
    print(f"JA:{fileJ} , {len(check_same_questions(fileJ, fileA))} , {check_same_questions(fileJ, fileA)}")
    print("\n")
    print(f"MJ:{fileM} , {len(check_same_questions(fileM, fileJ))} , {check_same_questions(fileM, fileJ)}")
    print("\n")
    print(f"HJ:{fileH} , {len(check_same_questions(fileH, fileJ))} , {check_same_questions(fileH, fileJ)}")
    print("\n")
    print(f"MH:{fileM} , {len(check_same_questions(fileM, fileH))} , {check_same_questions(fileM, fileH)}")
    print("\n")
    

MA:Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv , 14 , [['What does PS0KY mean by "file them down a bit"?', 'What does PS0KY mean by "file them down a bit"?', 4.0, 4.0], ['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 2.0, 2.0], ['How does this unclear part impact the overall meaning or intent of their previous statement?', 'How does this unclear part impact the overall meaning or intent of their previous statement?', 2.0, 2.0], ["To what extent does the lack of clarity in PS0KY's utterance affect the conversational trajectory?", "To what extent does the lack of clarity in PS0KY's utterance affect the conversational trajectory?", 2.0, 2.0], [nan, nan, 0.0, 0.0], ['How does this "Yeah." affect the subsequent QUD stack?', 'How does this "Yeah." affect the subsequent QUD stack?', 0.0, 0.0], ['Why did PS0L6 hesitate or pause with "mm . Let me just er" ?', 'Why did PS0L6 hesitate or pause with "m

## Check for all the file when they agreed (without subjectivity)

In [515]:
for fileA, fileM, fileH, fileJ in zip(file_path_axel, file_path_mario, file_path_hao, file_path_jinane):
    print(f"MA:{fileM} , {len(check_same_questions_without_subjectivity(fileM, fileA))} , {check_same_questions_without_subjectivity(fileM, fileA)}")
    print("\n")
    print(f"HA:{fileH} , {len(check_same_questions_without_subjectivity(fileH, fileA))} , {check_same_questions_without_subjectivity(fileH, fileA)}")
    print("\n")
    print(f"JA:{fileJ} , {len(check_same_questions_without_subjectivity(fileJ, fileA))} , {check_same_questions_without_subjectivity(fileJ, fileA)}")
    print("\n")
    print(f"MJ:{fileM} , {len(check_same_questions_without_subjectivity(fileM, fileJ))} , {check_same_questions_without_subjectivity(fileM, fileJ)}")
    print("\n")
    print(f"HJ:{fileH} , {len(check_same_questions_without_subjectivity(fileH, fileJ))} , {check_same_questions_without_subjectivity(fileH, fileJ)}")
    print("\n")
    print(f"MH:{fileM} , {len(check_same_questions_without_subjectivity(fileM, fileH))} , {check_same_questions_without_subjectivity(fileM, fileH)}")
    print("\n")
    

MA:Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv , 14 , [['What does PS0KY mean by "file them down a bit"?', 'What does PS0KY mean by "file them down a bit"?', 4.0, 4.0], ['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 2.0, 2.0], ['How does this unclear part impact the overall meaning or intent of their previous statement?', 'How does this unclear part impact the overall meaning or intent of their previous statement?', 2.0, 2.0], ["To what extent does the lack of clarity in PS0KY's utterance affect the conversational trajectory?", "To what extent does the lack of clarity in PS0KY's utterance affect the conversational trajectory?", 2.0, 2.0], [nan, nan, 0.0, 0.0], ['How does this "Yeah." affect the subsequent QUD stack?', 'How does this "Yeah." affect the subsequent QUD stack?', 0.0, 0.0], ['Why did PS0L6 hesitate or pause with "mm . Let me just er" ?', 'Why did PS0L6 hesitate or pause with "m

In [516]:
with open("same_questions_without_subjectivity_4.csv", "w") as f:
    f.write("File;Number of same questions;questions; \n")
    for fileA, fileM, fileH, fileJ in zip(file_path_axel, file_path_mario, file_path_hao, file_path_jinane):
        print(f"{fileM};{len(check_same_questions_without_subjectivity_4(fileM, fileA, fileH, fileJ))};{check_same_questions_without_subjectivity_4(fileM, fileA, fileH, fileJ)}")
        f.write(f"{fileM};{len(check_same_questions_without_subjectivity_4(fileM, fileA, fileH, fileJ))};{check_same_questions_without_subjectivity_4(fileM, fileA, fileH, fileJ)} \n")

Mario/deepseek/sheet_10_annotations_JAKIMOSKI.csv;9;[['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 2.0, 2.0, 2.0, 2.0], ['How does this unclear part impact the overall meaning or intent of their previous statement?', 'How does this unclear part impact the overall meaning or intent of their previous statement?', 'How does this unclear part impact the overall meaning or intent of their previous statement?', 'How does this unclear part impact the overall meaning or intent of their previous statement?', 2.0, 2.0, 2.0, 2.0], ["To what extent does the lack of clarity in PS0KY's utterance affect the conversational trajectory?", "To what extent does the lack of clarity in PS0KY's utterance affect the conversational trajectory?", "To what extent does the lack of clari

In [528]:
import ast
import pandas as pd


def convertir_en_liste(chaine):
    # Si la ligne est déjà une liste ou vide, on la laisse tranquille
    if not isinstance(chaine, str):
        return chaine

    try:
        # 1. On nettoie les sauts de ligne et espaces
        chaine_propre = chaine.strip()

        # 2. On remplace le 'nan' textuel par 'None' pour ast.literal_eval
        # On cible "nan" entouré de virgules ou crochets pour éviter de casser des mots
        chaine_propre = (
            chaine_propre.replace("[nan", "[None")
            .replace("nan]", "None]")
            .replace(" nan,", " None,")
            .replace(", nan", ", None")
        )

        # 3. On convertit en vraie liste Python
        return ast.literal_eval(chaine_propre)

    except Exception as e:
        # En cas de problème sur une ligne bizarre, on renvoie None (ou la chaîne d'origine)
        # pour éviter que tout le script ne se bloque
        print(f"Erreur sur la ligne : {chaine} -> {e}")
        return None

df = pd.read_csv("same_questions_without_subjectivity_4.csv", sep=";", encoding='utf-8-sig')
# On applique la fonction à toute la colonne 'questions'
df["questions"] = df["questions"].apply(convertir_en_liste)
print(df["Number of same questions"].sum())

382


## Check if there is error in the code

In [529]:
different = 0
for fileA, fileM, fileH, fileJ in zip(file_path_axel, file_path_mario, file_path_hao, file_path_jinane):
    for list in check_same_questions_without_subjectivity_4(fileM, fileA, fileH, fileJ):
        for i in range(0, len(list), 2):
            print(list)
            if (list[i] == list[i+1]) or (np.isnan(list[i]) == True & np.isnan(list[i+1]) == True):
                print("same")
            else:
                different += 1
print(f"Number of different questions: {different}")

['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 2.0, 2.0, 2.0, 2.0]
same
['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 2.0, 2.0, 2.0, 2.0]
same
['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 2.0, 2.0, 2.0, 2.0]
same
['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempti

In [530]:
different

0

In [ ]:
different = 0
for fileA, fileM, fileH, fileJ in zip(file_path_axel, file_path_mario, file_path_hao, file_path_jinane):
    for list in check_same_questions(fileM, fileA):
        for i in range(0, len(list), 2):
            print(list)
            if (list[i] == list[i+1]) or (np.isnan(list[i]) == True & np.isnan(list[i+1]) == True):
                print("same")
            else:
                different += 1

    for list in check_same_questions(fileH, fileA):
        for i in range(0, len(list), 2):
            print(list)
            if (list[i] == list[i+1]) or (np.isnan(list[i]) == True & np.isnan(list[i+1]) == True):
                print("same")
            else:
                different += 1
    
    for list in check_same_questions(fileJ, fileA):
        for i in range(0, len(list), 2):
            print(list)
            if (list[i] == list[i+1]) or (np.isnan(list[i]) == True & np.isnan(list[i+1]) == True):
                print("same")
            else:
                different += 1

    for list in check_same_questions(fileM, fileJ):
        for i in range(0, len(list), 2):
            print(list)
            if (list[i] == list[i+1]) or (np.isnan(list[i]) == True & np.isnan(list[i+1]) == True):
                print("same")
            else:
                different += 1

    for list in check_same_questions(fileH, fileJ):
        for i in range(0, len(list), 2):
            print(list)
            if (list[i] == list[i+1]) or (np.isnan(list[i]) == True & np.isnan(list[i+1]) == True):
                print("same")
            else:
                different += 1

    for list in check_same_questions(fileM, fileH):
        for i in range(0, len(list), 2):
            print(list)
            if (list[i] == list[i+1]) or (np.isnan(list[i]) == True & np.isnan(list[i+1]) == True):
                print("same")
            else:
                different += 1


['What does PS0KY mean by "file them down a bit"?', 'What does PS0KY mean by "file them down a bit"?', 4.0, 4.0]
same
['What does PS0KY mean by "file them down a bit"?', 'What does PS0KY mean by "file them down a bit"?', 4.0, 4.0]
same
['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 2.0, 2.0]
same
['What was PS0KY attempting to convey with their unclear utterance?', 'What was PS0KY attempting to convey with their unclear utterance?', 2.0, 2.0]
same
['How does this unclear part impact the overall meaning or intent of their previous statement?', 'How does this unclear part impact the overall meaning or intent of their previous statement?', 2.0, 2.0]
same
['How does this unclear part impact the overall meaning or intent of their previous statement?', 'How does this unclear part impact the overall meaning or intent of their previous statement?', 2.0, 2.0]
same
["To what extent does the lack of clarit

In [ ]:
different

0

# Extra Annotation with their score

## Files Paths

In [ ]:
extra_fp_a = "Extra_Annotation/tedQ_common_annotations_FELTEN.csv"
extra_fp_m = "Extra_Annotation/tedQ_common_annotations_mario.csv"
extra_fp_h = "Extra_Annotation/tedQ_common_annotations_Hao.csv"
extra_fps = [extra_fp_a, extra_fp_m, extra_fp_h]

json = "Extra_Annotation/tedQ_common.json"

## IAA with their score

In [ ]:
questions = []
with open(json, "r") as f:
    test = f.readlines()
    for question in test:
        if "text" in question:
            questions.append(question.replace('\t', "").replace('"text": "SPK:', ""))

In [ ]:
ratea = []
dfa = pd.read_csv(extra_fp_a, sep=None, engine='python', encoding='utf-8-sig')
for i in range(0, 32):
    ratea.append(int(dfa['rating'][i]))

ratem = []
dfm = pd.read_csv(extra_fp_m, sep=None, engine='python', encoding='utf-8-sig')
for i in range(0, 32):
    ratem.append(int(dfm['rating'][i]))

rateh = []
dfh = pd.read_csv(extra_fp_h, sep=None, engine='python', encoding='utf-8-sig')
for i in range(0, 32):
    rateh.append(int(dfh['rating'][i]))

extra_kappa_ma = cohen_kappa_score(ratea, ratem)
print(f"Kappa: between {extra_fp_a} and {extra_fp_m}: {extra_kappa_ma:.3f}")

extra_kappa_mh = cohen_kappa_score(ratem, rateh)
print(f"Kappa: between {extra_fp_m} and {extra_fp_h}: {extra_kappa_mh:.3f}")

extra_kappa_ah = cohen_kappa_score(ratea, rateh)
print(f"Kappa: between {extra_fp_a} and {extra_fp_h}: {extra_kappa_ah:.3f}")

Kappa: between Extra_Annotation/tedQ_common_annotations_FELTEN.csv and Extra_Annotation/tedQ_common_annotations_mario.csv: -0.075
Kappa: between Extra_Annotation/tedQ_common_annotations_mario.csv and Extra_Annotation/tedQ_common_annotations_Hao.csv: -0.018
Kappa: between Extra_Annotation/tedQ_common_annotations_FELTEN.csv and Extra_Annotation/tedQ_common_annotations_Hao.csv: 0.044


In [ ]:
with open("Extra_Annotation/Extra_Kappa.csv", "w") as f:
    f.write("Couple,Kappa\n")
    f.write(f"Mario-Axel,{extra_kappa_ma:.3f}\n")
    f.write(f"Mario-Hao,{extra_kappa_mh:.3f}\n")
    f.write(f"Axel-Hao,{extra_kappa_ah:.3f}\n")